In [ ]:
import jax
jax.config.update("jax_enable_x64", True)

In [ ]:
from astropy.io import fits
import os
import numpy as np
import jax.numpy as jnp

import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
tfb = tfp.bijectors

import matplotlib.pyplot as plt

In [ ]:
from gigalens.jax.scene import Component, Plane, LensModel
from gigalens.jax.profiles.mass.epl import EPL
from gigalens.jax.profiles.mass.shear import Shear
from gigalens.jax.profiles.mass.nfw import NFW_ELLIPSE, NFW_ELLIPSE_EINSTEIN
from gigalens.jax.profiles.mass.nfw_ellipse_slope import NFW_ELLIPSE_SLOPE
from gigalens.jax.profiles.mass.piemd import DPIE

from gigalens.jax.profiles.light.sersic import SersicEllipse
from gigalens.jax.profiles.light.shapelets import Shapelets

# from gigalens.jax.cosmo import wCDM_Cosmo

from gigalens.jax.scene_prob_model import Dataset, ImageData, ProbModel
from gigalens.simulator import SimulatorConfig

In [ ]:
from gigalens.jax.utils.grouped_priors import DiskEllipticity
from gigalens.jax.experimental.adaptive_supersample import AdaptiveImageData, plot_factor_map
from gigalens_research.plotting import plot_scene
from gigalens.jax.scene_simulator import SceneSimulator

In [ ]:
from ersatz_truth import truth_params, build_truth_model, simulate_ersatz

In [ ]:
import copy
orig_params = truth_params("improved_sersic_carousel.json")
orig_params

In [ ]:
p = copy.deepcopy(orig_params)

src1light = p['planes']['source1_2']['light']['source1']
src1light['n_sersic']=2.
# src1light['R_sersic']=
src1light['Ie']=6000.

src2light = p['planes']['source1_2']['light']['source2']
src2light['Ie']=3000.
src2light['R_sersic']= 0.2

src3light = p['planes']['source3']['light']['source3']
src3light['n_sersic']=0.7
src3light['R_sersic']=0.5
src3light['Ie']=200.

src4light = p['planes']['source4_5']['light']['source4']
src4light['Ie']=2500

src5light = p['planes']['source4_5']['light']['source5']
src5light['Ie']=5000

src9light = p['planes']['source9']['light']['source9']
src9light['Ie']=700.

src6light = p['planes']['source6']['light']['source6']
src6light['Ie']*=20

src7light = p['planes']['source7']['light']['source7']
src7light['Ie']*=20


src12light = p['planes']['source12_13']['light']['source12']
src12light['Ie']*=100

src13light = p['planes']['source12_13']['light']['source13']
src13light['Ie']*=40
src13light['n_sersic']=1.
src13light['R_sersic']=0.3

src8light = p['planes']['source8']['light']['source8']
src8light['Ie']*=10

src11light = p['planes']['source11']['light']['source11']
src11light['Ie']*=25
src11light['n_sersic'] = 2
# src11light = p['planes']['source11']['light']['source11']

In [ ]:

model = build_truth_model(p)
model.validate_params(p)                    # p is exactly this model's §5 layout

In [ ]:
obs = simulate_ersatz(model, p, "real_cutouts", seed=0, supersample=16)

In [ ]:
sims = [e.simulator(supersample=4) for e in obs]
figs = plot_scene(model, sims[8:], p, with_curves=False, scale='sqrt')
plt.show()

In [ ]:


cfg_k = SimulatorConfig(delta_pix=0.2, num_pix=300, supersample=1, kernel=None,
                       likelihood_precision="float64")
sims_nopsf = [SceneSimulator(model, cfg_k, sees=pl.light)
              for pl in model.planes if pl.has_light]
figs = plot_scene(model, sims_nopsf, p, with_curves=False, scale='sqrt')
plt.show()

In [ ]:
import copy
import ersatz_carousel_prior_improved as FIT   # the model you FIT with (lstsq, no Ie)

# Ascending redshift, so it lines up with `obs`; the assert is the guard, not the comment.
src_planes = [FIT.source1_2, FIT.source3, FIT.source4_5, FIT.source9, FIT.source7,
              FIT.source6, FIT.source12_13, FIT.source8, FIT.source11]
assert [e.plane for e in obs] == [pl.name for pl in src_planes], \
    [(e.plane, pl.name) for e, pl in zip(obs, src_planes) if e.plane != pl.name]

conservative_snr_levels = ((20.0, 8.0), (10.0, 4.0), (7.0, 2.0), (-np.inf, 1.0))
snr_levels = ((15.0, 8.0), (8.0, 4.0), (6.0, 2.0), (2.0, 1.0), (1.0, 0.5), (-np.inf, 0.25))

datasets = []
for i, (e, fit_plane) in enumerate(zip(obs, src_planes)):
    cfg = copy.deepcopy(e.sim_config)
    cfg.supersample = 1                      # the FIT grid, not the render grid
    datasets.append(AdaptiveImageData(
        e.image, cfg, exp_time=e.exp_time, background_rms=e.background_rms,
        sees=fit_plane.light,                # by identity: the fitting model's Components
        snr_levels=conservative_snr_levels if i in (4, 5, 6, 8) else snr_levels))

In [ ]:
from gigalens_research.astrometry import (
    Frame, PSFSpec, NoiseSpec, SystematicsBudget, measure_astrometry)

result = measure_astrometry(
    cutout,
    frame=Frame(transform_pix2angle=cfg.transform_pix2angle, ra_at_xy_0=-cfg.num_pix*cfg.delta_pix/2, dec_at_xy_0=-cfg.num_pix*cfg.delta_pix/2),
    psf=PSFSpec(kernel=cfg.kernel, supersampling_factor=4),
    noise=NoiseSpec(noise_map=sigma_map),
    init_ra=[-25., -6., 25.,],      # by-eye positions are fine
    init_dec=[6., -17., -15.,],
    search_radius=6.,
    systematics=SystematicsBudget(sigma_translation=1.5e-3),
)
print(result.summary())

from gigalens.jax.point_source_position import PointSourcePositionData

src11_ps_data = PointSourcePositionData(source_component, **result.to_gigalens_kwargs())

In [ ]:
filter_i = list(range(len(datasets)))
filtered_datasets = [datasets[i] for i in filter_i]

model_filtered = LensModel(
    [FIT.lens_plane, *[src_planes[i] for i in filter_i]],
    cosmo=FIT.cosmo, unconstrain="gaussian",
)
for d in filtered_datasets:
    plot_factor_map(d.adaptive_grid)

prob_model = ProbModel(model_filtered, filtered_datasets, mode="lstsq")

In [ ]:
from gigalens.jax.analysis import diagnose_undersampling

In [ ]:
import copy

def remove_key(pytree, key_to_remove):
    """
    Recursively copy a nested-dict pytree, dropping any dict entry
    whose key equals `key_to_remove` at any depth.
    """
    if isinstance(pytree, dict):
        return {
            k: remove_key(v, key_to_remove)
            for k, v in pytree.items()
            if k != key_to_remove
        }
    elif isinstance(pytree, (list, tuple)):
        cls = type(pytree)
        return cls(remove_key(v, key_to_remove) for v in pytree)
    else:
        # leaf node (e.g. a jax/np Array, float, etc.) -> copy as-is
        return copy.deepcopy(pytree)


# Example usage:
# new_tree = remove_key(original_tree)

rep = diagnose_undersampling(prob_model, remove_key(p, 'Ie'), reference_supersample=16, dataset_idxes=[0])

In [ ]:
for r in rep:
    r.plot()

In [ ]:
fig, axs = plt.subplots(1, len(datasets))
fig.set_size_inches(25, 3)
for ax, d in zip(axs, datasets):
    im = ax.imshow(d.image/d.error_map)
    fig.colorbar(im, ax=ax)

plt.show()

In [ ]:
from gigalens.jax.analysis.preflight import sampling_preflight

res = sampling_preflight(prob_model, p, run_gradient = False,
                       run_stiffness = False,
                       run_gates= False,)

In [ ]:
# print(res.summary())

In [ ]:
res.plot()

In [ ]:
from gigalens_research.inference_utils import *

z_truth = prob_model.unconstrained(p)
if jnp.any(~jnp.isfinite(z_truth)):
    raise ValueError("NAN IN Z")

ctx_in= InferenceContext.from_prob_model(prob_model)
pipeline = Pipeline(ctx_in, seed=0)

# def make_diag_qz(z_best):
#     return tfd.MultivariateNormalDiag(
#         loc=jnp.asarray(z_best),
#         scale_diag=jnp.full(z_best.shape[-1], 1e-3),
#     )

# pipeline.add(BridgeStage(
#     name="diag_qz_from_map",
#     version="v1",
#     requires=("z_best",),
#     produces=("qz",),
#     fn=make_diag_qz,
# ))

pipeline.add(SVIStage(
    num_steps=2000,
    n_vi=300,               
    init_scales=1e-3,       
    pbar_interval=5,
))

pipeline.add(MCLMCStage(
    n_chains=8,
    num_burnin_steps=20000,
    num_results=20000,
    desired_energy_variance=5e-4,
    seed=10,
    progress_bar=True,
    debug=True,
    regularize_mass_matrix=True,
))
# pipeline.add(MAMSStage(
#     n_chains=8,
#     num_burnin_steps=2000,
#     num_results=2000,
#     seed=10,
#     progress_bar=True,
#     debug=True,
#     regularize_mass_matrix=True,
# ))

In [ ]:
results_dir = os.path.join(
    os.path.expanduser("~"), "GIGALens-Code", "results",
    "ersatz_carousel_better", "startfit",
)
artifacts = pipeline.run(out_dir=results_dir,seed_artifacts={"z_best": z_truth}, resume=True)


In [ ]:
# import gigalens_research
# importlib.reload(gigalens_research.plotting)
from gigalens_research.plotting import PosteriorReport, PipelineReport
report = PosteriorReport(pipeline.posterior(), truth_x=p)
report.corner(plot_params=['cosmo/Om0', 'cosmo/w0', 'cosmo/wa'])#kind="cosmology")
report.convergence_panel(n_worst=5)

# report.source_comparison_panel()
# report.full_report()
plt.show()

In [ ]:
jnp.sum(pipeline.posterior().rhat > 1.01)

In [ ]:
report.image_panel()
report.source_panel()
plt.show()

In [ ]:
report.corner(kind="mass")
plt.show()

In [ ]:
import gigalens_research
importlib.reload(gigalens_research.plotting.diagnostics)
from gigalens_research.plotting import PosteriorReport, PipelineReport
pipeline_report = PipelineReport(pipeline)
fig = pipeline_report.diagnostics("mclmc", chain=3)
fig.show()

In [ ]:
pipeline_report.loss_histories()
plt.show()